   ... the structure you need to before you change the scheduler. The lecture
   starts ... introduces TASK 1, then explains thread organisation and interrupt
   behaviour. For your BSD work, the key points are where per-tick updates 
   belong and how every thread receives its initial state. The BSD formulas
   themselves come later. 

   TASK 1 HAS THREE PARTS: 
   - Priority Scheduling
   - Priority Donation
   - BSD Scheduler

   Priority scheduling lets more important threads receive CPU time first.
   Priority donation addresses the priority-inversion problem, which ... 
   BSD provides an alternative policy for how the system determines priority.

   These changes build on the existing thread system rather than replace the 
   whole operating system

   ... The lecturer strongly recommends individual tests during development
   because several MLFQS tests measure behaviour over minutes. He wans that a 
   full run can take roughly... so 

   the main files divide responsibillity between threads, synchronisation, and
   the timer. `thread.c` manages thread creation and execution, so most 
   scheduler changes belong there. `synch.c` implements synchronisation 
   primitives, including llocks and semaphores. `timer.c` handles...
   The lecturer explicitly adises you to keep thread-specific updates in the
   thread code. For your BSD implementation, this makes `thread_tick()` the 
   natural place to coordinate periodic scheduler updates ... 

   THE LIST LIBRARY AND DIAGNOSTIC HELPERS CAN EXPOSE MISTAKES EARLY. The
   lecturer describes an extra-list membership pointer that helps dtect an 
   element inserted twice into the same list. Duplicate 

   ...

   AN INTERRUPT HANDLER MUST REQUEST A CONTEXT SWITCH AT A SAFE POINT. Think of 
   it as a request to hand over the CPU once the urgent interrupt work finished. 


   When the hardware timer fires, the CPU enters "interrupt context", meaning
   it is performing critical, uninterruptbile hardware tasks and will 
   intentionally crash if you try to swap out the running thread by calling
   `thread_yield()` direclty. Instead, you must use `intr_yield_on_return()`,
   which acts like leaving a sticky note ... This is especially crucial  for the
   BSD scheduler because every timer tick actively recalculates thread 
   priorities, meaning a background thread might suddenly become the 
   highest-priority thread right in the middle of a hardware interrupt. 

   There is a classic chicken-and-egg problem when an OS boots: the ... 
   very first thread ... cannot be created using `thread_create`... because the
   OS threading system hasn't been built yet... 



   The 20-minute lecture segment introduces core concepts required for PintOS
   task 1 (scheduling)... the lecturer transitions from basic thread management
   into the mechanics of priority-based scheduling, detailing the catastrophic
   system failures ... that can occur when priorities are ignored (Priority
   Inversion) and how to engineer a solution (Priority Donation). Finally, the
   segment introduces the alternative 4.4BSD-style scheduler MLFQSS, which
   calculates priorities dynamically based on CPU load and requires simulating
   floating-point arithmeitc. 

   ... 
   YIELDING AND SLEEPING THREADS: ... whenever a thread is unblocked or wakes up
   from a timer sleep, the scheduler must immediately check if the newly 
   readied thread has a higher priority than the currently running thread. If it
   does, the current thread must yield.

   BASE vs. EFFECTIVE PRIORITY: ... Threads must be able to query and set their 
   own priorities. ... the distinction between a thread's base priority ...
   its own original, innate priority... and its effective priority ... its
   priority after receiving donations from other threads. 

   THE PRIORITY INVERSION PROBLEM: a detailed scenario: a Low-priority thread
   acquires a lock, a high-priority thread gets stuck waiting for that lock, and
   a medium-priority thread monopolizes the CPU ("infinite spotify download)...
   because the medium thread outranks the low thread, the low thread can
   never run to release the lock, effecitvely starving the high-priority thread...

   PRIORITY DNATION: To solve priority inversion, the high-priority thread must
   temporarily "donate" its priority to the low-priority thread holding the
   resouce. The lecturer details the necessary complexities of this system:
   handling multiple threads donating to a single lock holder, nested donations
   (Thread A waits on B, which waits on C) and correctly restoring the original
   base priority once the lock is released. 

   4.4BSD (MLFQS) SCHEDULER: As an alternaitve to strict priority scheduling, 
   the lecturer introduces a feedback scheduler that dynamically adjusts priorities
   based on real CPU usage and the system load average. Because these variables
   decay over time, they require decimal calculations. 

   ... As an alternative to strict priority scheduling, the lecturer introduces
   a feedback scheduler that dynamically adjusts priorities based on real CPU
   usage and the system load average. Because these variables decay over time,
   they require decimal calculations. 

 - Do not wake sleeping threads early...
 - Scope of Priority Modifications: Threads can only change their own base 
   priority using `thread_set_priority()`. They cannot directly change the 
   priorities of other threads. 
 - ONLY DONATE ON LOCKS: Priority donation should only be implemented for Locks,
   not Semaphores or Condition Variables. Locks have a strict, identifiable 
   owner (the thread that acquired it), whereas Semaphores do not, making it
   impossible to safely determine who to donate priority to. 

 - NESTING DEPTH LIMIT: While priority donations can nest (A -> B -> C), 
   the lecturer explicitly states that you can assume the nesting depth will 
   never exceeds 8 threads. You do not need to wite defensive code to 
   explicitly check if the depth has exceeded 8.
 - PROPER PRIORITY RESTORATION: When a lock is released and a donation is 
   revoked, the thread must revert exactly to its base priority (or its 
   highest remaining donation, if holding multiple locks). It should not revert
   to a generic default priority or to zero. 
 - No `#ifdef`s for the BSD scheduler: both the strict Priority Scheduler and
   the BSD scheduler must coexist in your compiled codebase. Do not use 
   compile-time `#ifdef` flags to switch between them. Instead, use the global
   boolean variable `thread_mlfqs` to switch routing logic at runtime. 


---
   The use of a global boolean variable (`thread_mlfqs`) instead of `#ifdef`
   comes down to how the autmted testing framework evaluates your codebase. 
   `#ifdef` is a compile-time directive, meaning the testing system would have 
   to compile two entirely separate veresions of your PintOS kernel to evaluate
   both schedulers. Instead, the graders require a single, unified compiled 
   kernel that contains the logic for both schedulers simultaneously. 

   ... no, PintoS does not dynamically switch schedulers mid-run. The scheduler
   choice is made exactly once at system startup. When the testing script
   boots your kernel, it can optinally pass the `-mlfqs` on the PintOS command
   line. During early initialization in `main()`, the OS parses these 
   command-line ooptions and sets the `thread_mlfqs` boolean accordingly. This
   architecture allows the test suite to rapidly rebooot the exact... 
---

 - SIMULATING FIXED-POINT MATH: The PintOS emulator does not have a Floating 
   Point Unit (FPU). You cannot use standard `float` or `double` C types for
   the BSD scheduler's load averages. You must simulate real arithmetic using 
   fixed-point integer mathematics. ... 


-- An extra list-membership poiinter prevents duplicate insertions by TRACKING
   WHETHER A NODE IS ALREADY A PART OF A LIST before allowing any insertion 
   operations to proce3ed. 


-- BSD: WHAT HAPPENS ON EACH TICK
   - CPU ACCOUNTING (`recent_cpu` increment): The scheduler increments the
     `recent_cpu` counter for the currently running thread by 1 on every timer
     tick to track how much processor time it has recently consumed.
   - PREEMPTION CHECK: The scheduler checks if the running thread has exhausted
     its allocated time slice (quantum) or if a higher-priority thread has 
     become ready. If so, it flags the thread for preemption.
   - PRIORITY RECALCULATION (Every 4 ticks): In classic 4.4BSD-derived 
     implementatins (such as PintOS)... thread priorities are reclaculated
     every fourth clock tick using the formula: 

     Priority = PRI_MAX - (recent_cpu / 4) - (nice * 2)

   - DECAY AND LOAD AVERAGES (Per Second): While `recent_cpu` updates on every
     tick, it decays once per second, and system-wide load averages (`load_avg`)
     are also recomputed once per second rather than every single tick.


---

   ... reduce average response time for running jobs on your system... 

   Like the priority scheduler, the advanced scheduler chooses the
   ... thread to run based on priorities. However, the advanced scheduler does 
   not do priority donation. Thus... we recommend that you hvae the priority
   ...

   You must write your code to allow us to choose a scheuling algorithm policy
   at PintOS ... with the `-mlfqs` kernel option. Passing this option sets
   `thread_mlfqs` declared in `threads/thread.h`

---

   When the 4.4BSD shceudler is enabled, threads no longer directly control
   their own priorities. The priority argument to `thread_create()`
   ... The priority argument to `thread_create()` should be ignored... as well 




---

   Our scheduler has 64 priorities and thus 64 readu queues, numbered 0 
   (PRI_MIN) through 63 (PRI_MAX). Lower numbers correspond to lower priorities,
   so that priority 0 is the lowest priority and ... Thread priority is
   calculated initially at thread initialization. It is also recalculated for
   each thread (if necessary) on every fourth clock tick. In either situation,
   it is determined by the formula:

   $$priority = PRI_MAX - (recent_cpu / 4) - (nice * 2)$$

   where `recent_cpu` is an estimate of the CPU time the thread has already used...
   result should be rounded down to the nearest integer (truncated)... The
   coefficients 1/4 and 2 on recent_cpu and nice... work well in practice but 
   lack deepter meaning... 

...
   instead, we use an exponentially weighted moving average, which takes this
   general form...

   x(0) = f(0)
   x(t) = ax(t - 1) + (1 - a)f(t)